### Street View Blurring System YOLOv8


In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl.metadata (10 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: fastjsonschema
    Found existing installation: fastjsonschema 2.21.1
    Uninstalling fastjsonschema-2.21.1:
      Successfully uninstalled fastjsonschema-2.21.1
  Attempting uninstall: typing-

In [2]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ultralytics
import cv2
from ultralytics import YOLO

# Verify system setup for training
ultralytics.checks()

Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M5 Pro)
Setup complete ✅ (15 CPUs, 24.0 GB RAM, 95.9/926.3 GB disk)


In [4]:
from pathlib import Path

base_path = Path("./data")

def check_dataset_consistency(base):
    splits = ['train', 'val', 'test']

    print(f"{'Split':<10} | {'Images':<10} | {'Labels':<10} | {'Status'}")
    print("-" * 50)

    for split in splits:
        # Correct pathing based on your screenshot:
        # Data/images/train/ and Data/labels/train/
        img_folder = base / 'images' / split
        lbl_folder = base / 'labels' / split

        # Count files (handling case sensitivity and common extensions)
        img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']
        img_count = sum(len(list(img_folder.glob(ext))) for ext in img_extensions)
        lbl_count = len(list(lbl_folder.glob('*.txt')))

        status = "✅ Match" if img_count == lbl_count and img_count > 0 else "❌ Mismatch or Empty"
        print(f"{split:<10} | {img_count:<10} | {lbl_count:<10} | {status}")

check_dataset_consistency(base_path)

Split      | Images     | Labels     | Status
--------------------------------------------------
train      | 23604      | 23604      | ✅ Match
val        | 1073       | 1073       | ✅ Match
test       | 386        | 386        | ✅ Match


In [5]:
#converting YOLO coordinates to pixel coordinates

import cv2
import matplotlib.pyplot as plt

def audit_sample(image_path, label_path):
    # Load image
    img = cv2.imread(str(image_path))
    h, w, _ = img.shape

    # Read YOLO label (class, cx, cy, bw, bh)
    with open(label_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        data = line.split()
        cx, cy, bw, bh = map(float, data[1:])

        # BUSINESS LOGIC: Convert normalized to pixel coordinates
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)

        # Draw green bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, "License Plate", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Data Audit: Bounding Box Verification")
    plt.axis('off')
    plt.show()

# Test on one image from your 25,470 training set
# Update filenames to real ones in your folder
# audit_sample(data_splits['train']/'images'/'example.jpg', data_splits['train']/'labels'/'example.txt')

## Creating config yaml

In [7]:
import yaml

# Use .resolve() to get the full absolute path on your Mac
abs_base = base_path.resolve()

yaml_content = {
    'train': str(abs_base / 'images' / 'train'),
    'val': str(abs_base / 'images' / 'val'),
    'test': str(abs_base / 'images' / 'test'),
    'nc': 1,
    'names': ['license_plate']
}

with open('license_plate.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print("✅ license_plate.yaml created successfully.")

✅ license_plate.yaml created successfully.


## Training phase

In [8]:
 from ultralytics import YOLO

# 1. Load the Model
# We use 'yolov8n.pt' (Nano) because it balances speed and accuracy
# and is less likely to overfit on our reduced dataset of ~2,100 images.
model = YOLO('yolov8n.pt')

# 2. Train the Model
results = model.train(
    data='license_plate.yaml',    # The config file we created earlier
    epochs=50,                   # 50 iterations to allow the model to learn features
    imgsz=640,                   # Resize images to 640x640 for consistency
    batch=16,                    # Process 16 images at a time
    device='mps',                # Change to 'cpu' if not on Apple Silicon
    name='plate_anonymizer_v1'   # Folder name for your logs and weights
)

/Users/kishankunal/workspace/AppliedAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M5 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=license_plate.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_anonymizer_v13, nbs=64, nms=False, op

# Evaluation & Performance Metrics

In [2]:
# This will generate a 'results.csv' and confusion matrix in 'runs/detect/val'
from ultralytics import YOLO

# Load your specific trained weights
# Replace the path below with the actual path to your best.pt
model = YOLO('/YOLO8-LicensePlateDetectionBlurring/runs/detect/plate_anonymizer_v13/weights/best.pt')

# Now run your validation code
metrics = model.val(data='license_plate.yaml')
print(f"mAP@50-95: {metrics.box.map}")

Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M5 Pro)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 865.0±924.7 MB/s, size: 258.7 KB)
val: Scanning /Users/kishankunal/workspace/AppliedAI/YOLO8-LicensePlateDetectionBlurring/data/labels/val.cache... 1073 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1073/1073 132.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 1.3s/it 1:281.3sss
                   all       1073       1573      0.867      0.786      0.844      0.453
Speed: 0.1ms preprocess, 77.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /Users/kishankunal/workspace/AppliedAI/runs/detect/val2
mAP@50-95: 0.45311050933547214


# Post-Processing

In [3]:
import cv2
import os
from pathlib import Path

# Now your code will work:
test_images = list(Path("./data/images/test").glob("*.jpg"))

def process_and_blur(image_path, output_folder, model):
    img = cv2.imread(image_path)
    results = model(img)[0]

    for box in results.boxes:
        # Confidence check as per your document
        if box.conf > 0.4:
            # Get pixel coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Extract ROI (Region of Interest)
            plate_region = img[y1:y2, x1:x2]

            # Apply Gaussian Blur (Privacy Compliance)
            # Higher numbers in (k, k) = more blur
            blurred_plate = cv2.GaussianBlur(plate_region, (35, 35), 0)

            # Overwrite original area
            img[y1:y2, x1:x2] = blurred_plate

    # Save the redacted output
    filename = os.path.basename(image_path)
    cv2.imwrite(os.path.join(output_folder, f"redacted_{filename}"), img)

# Apply to your TEST folder
test_images = list(Path("./data/images/test").glob("*.jpg"))
os.makedirs("output_redacted", exist_ok=True)

for img_p in test_images:
    process_and_blur(str(img_p), "output_redacted", model)


0: 576x640 1 license_plate, 22.9ms
Speed: 2.8ms preprocess, 22.9ms inference, 0.3ms postprocess per image at shape (1, 3, 576, 640)

0: 512x640 2 license_plates, 18.6ms
Speed: 1.1ms preprocess, 18.6ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 640)

0: 480x640 1 license_plate, 16.7ms
Speed: 0.9ms preprocess, 16.7ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 license_plate, 17.8ms
Speed: 1.1ms preprocess, 17.8ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 640x480 1 license_plate, 17.3ms
Speed: 1.1ms preprocess, 17.3ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 480)

0: 480x640 3 license_plates, 17.9ms
Speed: 1.0ms preprocess, 17.9ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 license_plate, 18.7ms
Speed: 1.0ms preprocess, 18.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 448x640 1 license_plate, 17.3ms
Speed: 0.9ms preprocess, 17.3ms

# Measuring Evaluation Metrics

In [4]:
import time
import numpy as np

def evaluate_performance(model, test_images_path):
    latencies = []

    # Measure Inference Latency
    for img_path in test_images_path[:50]:  # Sample 50 images
        start_time = time.time()
        model.predict(img_path, verbose=False)
        end_time = time.time()
        latencies.append((end_time - start_time) * 1000)  # Convert to ms

    avg_latency = np.mean(latencies)
    print(f"--- Performance Report ---")
    print(f"Average Inference Latency: {avg_latency:.2f} ms per frame")
    print(f"Throughput: {1000/avg_latency:.2f} FPS")

# Run the evaluation
evaluate_performance(model, test_images)

--- Performance Report ---
Average Inference Latency: 22.16 ms per frame
Throughput: 45.12 FPS


# Edge Deployment (Exporting)

In [6]:
import sys
!{sys.executable} -m pip install onnx onnxslim

  Using cached onnx-1.19.1-cp39-cp39-macosx_12_0_universal2.whl.metadata (7.0 kB)
  Using cached onnxslim-0.1.87-py3-none-any.whl.metadata (10 kB)
  Using cached ml_dtypes-0.5.4-cp39-cp39-macosx_10_9_universal2.whl.metadata (8.9 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
Using cached onnx-1.19.1-cp39-cp39-macosx_12_0_universal2.whl (18.3 MB)
Using cached onnxslim-0.1.87-py3-none-any.whl (235 kB)
Using cached ml_dtypes-0.5.4-cp39-cp39-macosx_10_9_universal2.whl (676 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [onnxslim]2/4 [onnx]


In [8]:

path = model.export(format='onnx')
print(f"✅ Success! Your production-ready model is at: {path}")

Ultralytics 8.4.11 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M5 Pro)

PyTorch: starting from '/Users/kishankunal/workspace/AppliedAI/runs/detect/plate_anonymizer_v13/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)

ONNX: starting export with onnx 1.19.1 opset 22...
ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 0.4s, saved as '/Users/kishankunal/workspace/AppliedAI/runs/detect/plate_anonymizer_v13/weights/best.onnx' (11.7 MB)

Export complete (0.4s)
Results saved to /Users/kishankunal/workspace/AppliedAI/runs/detect/plate_anonymizer_v13/weights
Predict:         yolo predict task=detect model=/Users/kishankunal/workspace/AppliedAI/runs/detect/plate_anonymizer_v13/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/Users/kishankunal/workspace/AppliedAI/runs/detect/plate_anonymizer_v13/weights/best.onnx imgsz=640 data=license_plate.yaml  
Visualize:       https://netron.app
✅ Success! Your production-ready 

# Continuous Learning (Feedback Loop)
This script implements your recommendation for a Feedback Loop. It identifies "Low-Confidence" detections (e.g., between 25% and 50% confidence) and saves them to a specific folder for human review.

In [9]:
def feedback_loop_filter(img_path, model, review_folder="needs_review"):
    os.makedirs(review_folder, exist_ok=True)
    results = model(img_path)[0]

    for box in results.boxes:
        conf = box.conf.item()
        # STRATEGIC LOGIC: Flag low-confidence detections for human review
        if 0.25 < conf < 0.50:
            print(f"⚠️ Low confidence ({conf:.2f}) detected for {img_path}. Flagging for review.")
            filename = os.path.basename(img_path)
            # Copy or save image to review folder
            cv2.imwrite(f"{review_folder}/review_{filename}", results.orig_img)

In [10]:
import os
import cv2
from pathlib import Path

# 1. Define Confidence Zones
CONFIDENCE_THRESHOLD = 0.50  # Anything above this is blurred automatically
REVIEW_ZONE_START = 0.25    # Anything between 0.25 and 0.50 goes to Review

# 2. Setup Folders
os.makedirs("output_redacted", exist_ok=True)
os.makedirs("needs_human_review", exist_ok=True)

def test_feedback_loop(image_list, model):
    for img_path in image_list:
        img = cv2.imread(str(img_path))
        results = model(img, verbose=False)[0]

        needs_review = False
        detections_found = False

        for box in results.boxes:
            conf = box.conf.item()
            detections_found = True

            # CASE A: High Confidence -> Redact
            if conf >= CONFIDENCE_THRESHOLD:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                roi = img[y1:y2, x1:x2]
                img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (35, 35), 0)

            # CASE B: Low Confidence -> Flag for Feedback Loop
            elif REVIEW_ZONE_START <= conf < CONFIDENCE_THRESHOLD:
                needs_review = True

        # Save to appropriate folder
        filename = img_path.name
        if needs_review:
            cv2.imwrite(f"needs_human_review/REVIEW_{filename}", img)
            print(f"⚠️ {filename}: Flagged for Human Review (Conf: {conf:.2f})")
        elif detections_found:
            cv2.imwrite(f"output_redacted/REDACTED_{filename}", img)
            print(f"✅ {filename}: Successfully Redacted.")
        else:
            print(f"ℹ️ {filename}: No plates detected.")

# Run the test
test_images = list(Path("./data/images/test").glob("*.jpg"))
test_feedback_loop(test_images, model)

✅ 0f0596b1c511e071.jpg: Successfully Redacted.
⚠️ 791d63b1c3499e4f.jpg: Flagged for Human Review (Conf: 0.29)
✅ 5daafbcf76fa6602.jpg: Successfully Redacted.
✅ defd8f4b30b3e1e1.jpg: Successfully Redacted.
⚠️ d6c5271e96ec1a61.jpg: Flagged for Human Review (Conf: 0.45)
⚠️ 8256e277c7f47797.jpg: Flagged for Human Review (Conf: 0.34)
✅ a774a6f81fea258b.jpg: Successfully Redacted.
✅ 453a77009b27e253.jpg: Successfully Redacted.
✅ 15a51e29f5ceffd8.jpg: Successfully Redacted.
✅ efde2a25c1c9c924.jpg: Successfully Redacted.
✅ 2b97f5bf137ee8d1.jpg: Successfully Redacted.
✅ 4df1448703257ff0.jpg: Successfully Redacted.
✅ 895d440e05b2a8d8.jpg: Successfully Redacted.
✅ d565d93637d4e76d.jpg: Successfully Redacted.
✅ 03b7b71e1ffcb7a8.jpg: Successfully Redacted.
✅ d830c3573e57bfc0.jpg: Successfully Redacted.
⚠️ ce97f7bc90e97109.jpg: Flagged for Human Review (Conf: 0.44)
⚠️ d5f4069e6734ac06.jpg: Flagged for Human Review (Conf: 0.35)
✅ 3e2306f2cf4b2c67.jpg: Successfully Redacted.
✅ 593e594137f374ab.jpg: Suc